# **Nebula Core** — serve + tunnel (free T4)

Run this AFTER the training notebook (same session, or any new session with the
GGUF saved in Google Drive). It:

1. Locates your trained GGUF (session folder → Google Drive fallback)
2. Saves it to Drive so it is never lost when the session ends
3. Serves it as an OpenAI-compatible API on the T4 GPU (`/v1/chat/completions`)
4. Opens a public HTTPS tunnel (cloudflared) and prints the URL
5. Self-tests one real Nebula-style build prompt through YOUR model

At the end, paste the printed `LLM_CUSTOM_BASE_URL` into the Nebula chat —
the platform will wire it in and verify live. While the tunnel is offline,
Nebula automatically runs on the free Workers AI engine — zero downtime.

In [ ]:
# ── 1. Locate the trained GGUF ─────────────────────────────────
import glob, os

CANDIDATES = [
    "/content/nebula-core/gguf",                     # same-session training output
    "/content/drive/MyDrive/nebula-core",            # persisted from a past session
]

GGUF = None
for d in CANDIDATES:
    hits = glob.glob(os.path.join(d, "*q4_k_m*.gguf")) or glob.glob(os.path.join(d, "*q5_k_m*.gguf"))
    if hits:
        GGUF = hits[0]
        print(f"found: {GGUF}  ({os.path.getsize(GGUF)/1e9:.2f} GB)")
        break

if not GGUF:
    raise RuntimeError(
        "No GGUF found.\n"
        " A) If this is the SAME session that just trained: re-run the export cell\n"
        "    of the training notebook (it writes /content/nebula-core/gguf).\n"
        " B) If that session ended: re-run the training notebook (Run all, ~90 min) —\n        "    it now auto-saves the GGUF to Google Drive, so this never happens again.\n"
        " C) Or upload your downloaded nebula-core-q4_k_m.gguf to /content/ and set\n",
        "    GGUF = '/content/nebula-core-q4_k_m.gguf' manually, then skip this error."
    )
MODEL_DIR = os.path.dirname(GGUF)

In [ ]:
# ── 2. Persist to Google Drive (one-time, ~1 min) ──────────────
# So the model survives Colab session ends and future serving sessions
# can start straight from Drive.
import glob, os, shutil
if not GGUF.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    dst_dir = '/content/drive/MyDrive/nebula-core'
    os.makedirs(dst_dir, exist_ok=True)
    for f in glob.glob(os.path.join(MODEL_DIR, '*.gguf')):
        dst = os.path.join(dst_dir, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)
            print('saved ->', dst)
    print('Drive copy complete:', os.listdir(dst_dir))
else:
    print('already on Drive — nothing to copy')

In [ ]:
# ── 3. Runtime: llama.cpp (CUDA) + cloudflared ─────────────────
# Tries the official prebuilt CUDA binary first (~30 s); falls back to
# compiling llama-cpp-python with CUDA (~10-15 min) if the artifact 404s.
import json, subprocess, os

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)

LLAMA_SERVER = None

# (a) prebuilt llama.cpp CUDA binary
rel = sh('curl -fsSL https://api.github.com/repos/ggml-org/llama.cpp/releases/latest').stdout
try:
    assets = [a['browser_download_url'] for a in json.loads(rel).get('assets', [])
              if 'ubuntu' in a['name'] and 'cuda' in a['name'] and a['name'].endswith('.zip')]
    if assets:
        url = assets[0]
        print('downloading prebuilt CUDA llama.cpp:', url.rsplit("/",1)[-1])
        sh(f'curl -fsSL -o /tmp/llama.zip "{url}" && unzip -o -q /tmp/llama.zip -d /tmp/llamacpp')
        exe = sh('ls /tmp/llamacpp/*/llama-server /tmp/llamacpp/llama-server 2>/dev/null | head -1').stdout.strip()
        if exe and os.path.exists(exe):
            sh(f'chmod +x {exe} && ln -sf {exe} /usr/local/bin/llama-server')
            LLAMA_SERVER = 'llama-server'
            print('prebuilt llama-server ready')
except Exception as e:
    print('prebuilt route skipped:', e)

# (b) fallback: pip CUDA compile
if not LLAMA_SERVER:
    print('compiling llama-cpp-python with CUDA (10-15 min, one-time per session)…')
    r = sh('CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python')
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError('llama-cpp-python install failed')
    LLAMA_SERVER = 'python -m llama_cpp.server'
    print('llama-cpp-python ready')

# cloudflared (static binary, always works)
r = sh('curl -fsSL -o /usr/local/bin/cloudflared '
       'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
       '&& chmod +x /usr/local/bin/cloudflared')
if r.returncode != 0:
    print(r.stderr[-500:]); raise RuntimeError('cloudflared download failed')
print('cloudflared ready:', sh('cloudflared --version').stdout.strip())

In [ ]:
# ── 4. Serve the model (OpenAI-compatible on :8080, T4 GPU) ────
# Optional hardening: uncomment the API_KEY lines to require a Bearer key.
import subprocess, time, requests, os

# API_KEY = 'nebula-' + os.urandom(12).hex()   # uncomment to require a key

try:
    subprocess.run('pkill -f llama-server', shell=True, capture_output=True)
except Exception:
    pass

if LLAMA_SERVER == 'llama-server':
    cmd = (f'nohup llama-server -m "{GGUF}" --port 8080 --host 0.0.0.0 '
           f'--ctx-size 8192 --parallel 2 --alias nebula-core '
           f'> /tmp/llama.log 2>&1 &')
    # to require auth instead add:  --api-key "$API_KEY"
else:
    cmd = (f'nohup python -m llama_cpp.server --model "{GGUF}" '
           f'--host 0.0.0.0 --port 8080 --n_ctx 8192 '
           f'> /tmp/llama.log 2>&1 &')
    # to require auth instead add:  --api_key "$API_KEY"
subprocess.run(cmd, shell=True)

for i in range(90):
    try:
        if requests.get('http://127.0.0.1:8080/health', timeout=2).status_code == 200:
            print(f'model server healthy after {i*2}s (GPU)'); break
    except Exception:
        pass
    time.sleep(2)
else:
    print('health check slow — tail of log:'); print(open('/tmp/llama.log').read()[-1200:])

In [ ]:
# ── 5. Public HTTPS tunnel (cloudflared) ───────────────────────
import subprocess, re, time

subprocess.run('pkill -f cloudflared', shell=True, capture_output=True)
subprocess.run('nohup cloudflared tunnel --url http://localhost:8080 '
               '> /tmp/tunnel.log 2>&1 &', shell=True)

TUNNEL_URL = None
for i in range(45):
    time.sleep(2)
    log = open('/tmp/tunnel.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        TUNNEL_URL = m.group(0)
        break
if not TUNNEL_URL:
    print('tunnel log tail:'); print(open('/tmp/tunnel.log').read()[-1200:])
    raise RuntimeError('tunnel did not start')
print('PUBLIC URL:', TUNNEL_URL)
print()
print('LLM_CUSTOM_BASE_URL =', TUNNEL_URL + '/v1')
print('LLM_CUSTOM_MODEL    = nebula-core')
print()
print('>>> Paste the two lines above into the Nebula chat. <<<')
print('>>> Keep this Colab tab open while serving. <<<')

In [ ]:
# ── 6. Self-test: one REAL Nebula build prompt through YOUR model ─
import requests, time

msgs = [
    {"role": "system", "content": "You are Nebula Core, the fine-tuned engineering model of the Nebula platform. You build production-quality websites on the FIRST round. Output exactly the format asked."},
    {"role": "user", "content": "Section: cta. Requirement: a closing CTA band: one promise, one button, no clutter. Business: dental clinic group. Brand tokens: bg #faf7f2, ink #191919, accent #0e7c66. Output the complete <section> with scoped <style>."},
]
t0 = time.time()
r = requests.post('http://127.0.0.1:8080/v1/chat/completions', json={
    'model': 'nebula-core', 'messages': msgs,
    'temperature': 0.4, 'max_tokens': 700,
}, timeout=300)
out = r.json()['choices'][0]['message']['content']
secs = time.time() - t0
print(f'generated {len(out)} chars in {secs:.1f}s  (~{len(out.split())/max(secs,0.1):.1f} tok/s)')
print('--- first 900 chars ---')
print(out[:900])
assert '<section' in out.lower(), 'model did not emit a section — check training'
print()
print('SELF-TEST PASSED — this is your own model, building, on your own weights.')